# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam271/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My lane as an ML task

**Task type: Scoring**

My lane is Refresh / Content Opportunity Scoring. The goal is to assign each content page a score that represents its priority for human review. A higher score would indicate that the page shows more of the observed signals associated with the provisional refresh opportunity. Scoring fits this problem because the output needs to prioritize many pages, rather than only classify them as yes or no. The final score is intended for directional decision-support, not an automatic refresh decision.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print("Task type: Scoring")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Task type: Scoring
Rows: 23,366
Columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter dataset, I will use `trend_direction == "down"` as a **provisional proxy** for the scoring task. This label comes from an observed trend field in the starter data and is not a causal label for whether a refresh will work.

The proxy helps define an initial learning problem: identify pages that resemble pages with an observed downward trend. In later work, I can test stronger future-outcome labels if the available data supports them.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_proxy"] = (df["trend_direction"] == "down").astype(int)

print("Provisional proxy: trend_direction == 'down'")
print(f"Proxy-positive pages: {df['is_declining_proxy'].sum():,}")
print(f"Proxy-positive rate: {df['is_declining_proxy'].mean():.3f}")

Provisional proxy: trend_direction == 'down'
Proxy-positive pages: 12,641
Proxy-positive rate: 0.541


## 3. Success metric

*One metric you can defend. What number means 'good'?*



**Success metric: Precision@50**

The practical use case is prioritization: a reviewer may only have time to inspect a limited number of pages. Precision@50 measures how many of the top 50 scored pages match the provisional target.

As an initial benchmark, the hand-written rule from the earlier starter-model experiment achieved **0.680 Precision@50**. A future scoring approach would need to beat this baseline under the same validation setup to demonstrate useful improvement. This metric focuses on the quality of the highest-priority recommendations rather than overall accuracy.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_precision_at_50 = 0.680

print("Success metric: Precision@50")
print(f"Starter hand-rule baseline Precision@50: {baseline_precision_at_50:.3f}")
print("Future scoring approach should be evaluated against this baseline.")

Success metric: Precision@50
Starter hand-rule baseline Precision@50: 0.680
Future scoring approach should be evaluated against this baseline.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## The unit of analysis

**One row = one content page.**

Each row represents one anonymized content item and contains observable search/content signals that can be used to understand its current state and prioritize it for review. The model output would therefore produce one score per content page.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the actual unit of analysis.
page_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
]

available_columns = [c for c in page_columns if c in df.columns]

page_sample = df[available_columns].head(10)

print("Unit of analysis: one content page")
print(f"Rows shown: {len(page_sample)}")
display(page_sample)

Unit of analysis: one content page
Rows shown: 10


,content_id,client_id,trend_direction,impressions_90d,ctr,avg_position,content_age_days
0,content_304f48230142,client_f369cb89fc,down,3803.0,0.76,10.6,187.0
1,content_a1fb4e703a9e,client_4e07408562,down,15320.0,0.05,20.3,445.0
2,content_9aa793d4d895,client_7f2253d7e2,down,12581.0,0.09,36.5,141.0
3,content_331d6c4de07b,client_19581e27de,stable,11751.0,0.49,6.2,463.0
4,content_d99b7a2d90ca,client_3fdba35f04,down,19140.0,0.13,44.0,263.0
5,content_d4084a4bc775,client_f369cb89fc,down,3970.0,0.03,8.5,147.0
6,content_9a34b442b552,client_8722616204,down,20.0,0.00,7.0,90.0
7,content_a63219c6e95a,client_19581e27de,stable,1724.0,0.06,21.2,445.0
8,content_5e6c160719bc,client_6208ef0f77,down,32574.0,0.09,46.0,90.0
9,content_c27558df2b0c,client_19581e27de,down,1240.0,0.16,4.9,257.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


A fixed rule can prioritize pages using one or a few manually chosen thresholds, but the available data contains several signals that can interact, including impressions, CTR, average position, content age, and freshness-related information.

ML can learn combinations of these observed signals and produce a continuous priority score instead of relying on one fixed cutoff. The earlier experiments provide evidence that a learned model can outperform the simple hand rule on the measured Precision@50 metric under the tested validation setup.

This does not mean ML proves causation or predicts Google's algorithm. It means the model can be tested as a decision-support ranking method using observed data.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
signals = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
]

available_signals = [c for c in signals if c in df.columns]

print("Candidate observable signals:")
for col in available_signals:
    print(f"- {col}")

print("\nReason: multiple signals can be combined into a priority score.")
print("Claim boundary: this is decision-support, not causal proof.")

Candidate observable signals:
- impressions_90d
- ctr
- avg_position
- content_age_days

Reason: multiple signals can be combined into a priority score.
Claim boundary: this is decision-support, not causal proof.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.